# <span style="color: magenta">**Instrumento**</span>: controladores de sintetizador

Un instrumento es esencialmente un *controlador* para producir notas con un modelo de síntesis:

- Cada nota pedida genera un nuevo sintetizador
    
    - que se destruye cuando acaba la nota, i.e., cuando se completa la envolvente ADSR 


- Un instrumento es un **generador de señal**: incorpora el método *next()* para producir señal bajo demanda

    - Incorpora métodos *noteOn* y *noteOff* para lanzar y parar notas

    - Contiene los parámetros del sintetizador (el modelo de síntesis), pero NO el sintetizador en sí. Esto implica:

        - Pueden cambiarse dinámicamente los parámetros de síntesis para cada nota

        - Podrían cambiarse incluso los parámetros mientras suena una nota (no implementado en este modelo)

- Nuestro instrumento será polifónico: incorpora un **diccionario de canales** donde se conectan los sintetizadores activos.        

    - Cuando se lanza una nota, el sintetizador creado se vincula a un canal, identificado por la nota (midi) que lo ha lanzado

        - Si se *relanza* la misma nota, se para la primera y se inicia la nueva... similar a un piano (podría cambiarse este comportamiento)

- El *método next()* se comporta como un **mezclador**: suma la señal de todos los canales activos:

    - La mezcla resultante se devuelve en el método next

    - Hace *limpieza* de sintetizadores inactivos: elimina los canales *off* (ya terminados)


#### Las notas se identifican por su número MIDI 

- LA-4 es la nota midi 69 y suena a 440 Hz. Podemos 

- Partiendo de esa nota referencia obtenemos la tabla de frecuencias (temperadas) de las notas midi (de 0 a 127). En el módulo *const.py* tenemos:
    
    $$freqsMidi = [ 440*2^{i/12}\ para\ i\in [-69,58]]$$

    Los exponentes negativos bajan el pitch de las notas

    En Python: ```freqsMidi = [ 440*2.0**(i/12.0) for i in range(-69,58)]```

- El C4 (do central) es la nota 60 con freq 261.62 Hz


<center>
<img src="media/midi-freqs.jpg" width="1400" />
</center>

    



In [ ]:
freqsMidi = [ 440*2.0**(i/12.0) for i in range(-69,58)]
print(freqsMidi[60])
print(freqsMidi[69])


## Inciso: Sliders personalizados en TkInter para interactuar con más facilidad

- Creamos una clase *Slider* para simplificar (similar a los controladores interactivos que vimos para notebooks). 

- El slider 

    - define el nombre, el valor inicial, el rango de valores,... 
    
    - tiene un método *get* para obtener el valor del slider

    - el propio slider queda asociado al parámetro que controla
  


In [ ]:
%%writefile slider.py

import sys
sys.path.insert(0, "files") 

import numpy as np    
import matplotlib.pyplot as plt
from tkinter import *

class Slider:
    def __init__(self,tk,name='a',ini=0.2,from_=0.0,to=1.0,step=0.1,orient=HORIZONTAL,packSide=TOP):
        self.val = DoubleVar() # para guardar y modificar el valor del parámetro en cuestión
        self.val.set(ini)
                
        self.scale = Scale(tk, label=name,
            from_=from_, to=to, resolution=step, 
            orient=orient, width=30,sliderlength=10, length=300, bd=6,
            variable = self.val)
            
        self.scale.pack(side=packSide)

    def get(self):
        return self.val.get()


In [ ]:
import sys
sys.path.insert(0, "files") 

# prueba de slider
from slider import *
from tkinter import *


tk = Tk()
s = Slider(tk,name='MiControl')
tk.mainloop()

# Instrumentos


- Utilizamos el sintetizador FM como modelo de síntesis

- Definimos sliders para controlar

    - Los parámetros de la envolvente ADSR: *attack*, *decay*, *sustain*, *release*

    - Los parámetros de síntesis: *ratio*, *beta*, *amplitud*

    - La frecuencia correspondiente al *carrier*, i.e., *el pitch de la nota vendrá controlado mediante pulsaciones de teclado*

    - La nota se mantiene mientras está pulsada la nota y se lanza el noteOff cuando se suelta (se reproduce el *release*)


In [ ]:
%%writefile files/instrument.py

import sys
sys.path.insert(0, "./files")        


import numpy as np   
import matplotlib.pyplot as plt
from consts import *
from tkinter import *
from slider import *
from adsr import *
from synthFM import *

class Instrument:
    def __init__(self,tk,name="Control FM synthetizer",amp=0.2,ratio=3,beta=0.6): 
        
        frame = LabelFrame(tk, text=name, bg="#808090")
        frame.pack(side=LEFT)
        
        # Synth params con sus sliders
        frameOsc = LabelFrame(frame, text="FM oscillator", bg="#808090")
        frameOsc.pack(side=LEFT, fill="both", expand="yes")
        
        self.ampS = Slider(frameOsc,'amp',packSide=TOP,
                           ini=amp,from_=0.0,to=1.0,step=0.05) 

        self.ratioS = Slider(frameOsc,'ratio',packSide=TOP,
                           ini=ratio,from_=0.0,to=20.0,step=0.5)
    
        self.betaS = Slider(frameOsc,'beta',packSide=TOP,
                            ini=beta,from_=0.0,to=10.0,step=0.05) 
        
        # una ventana de texto interactiva para poder lanzar notas con el teclado del ordenador
        text = Text(frameOsc,height=4,width=40)
        text.pack(side=BOTTOM)
        text.bind('<KeyPress>', self.down)
        text.bind('<KeyRelease>', self.up)

       
        # ADSR params con sus sliders
        frameADSR = LabelFrame(frame, text="ADSR", bg="#808090")
        frameADSR.pack(side=LEFT, fill="both", expand="yes", )
        self.attackS = Slider(frameADSR,'attack',
                           ini=0.01,from_=0.0,to=0.5,step=0.005,orient=HORIZONTAL,packSide=TOP) 

        self.decayS = Slider(frameADSR,'decay',
                           ini=0.01,from_=0.0,to=0.5,step=0.005,orient=HORIZONTAL,packSide=TOP)

        self.sustainS = Slider(frameADSR,'sustain',
                   ini=0.4,from_=0.0,to=1.0,step=0.01,orient=HORIZONTAL,packSide=TOP) 
                    
        self.releaseS = Slider(frameADSR,'release',
                   ini=0.5,from_=0.0,to=4.0,step=0.05,orient=HORIZONTAL,packSide=TOP) 


        
        # canales indexados por la nota de lanzamiento -> solo una nota del mismo valor
        self.channels = dict()     

        # diccionario de tails para guardar las notas apagadas pendientes de terminación (release)
        self.tails = dict()
                         

    # obtenemos todos los parámetros del sintetizador (puede servir para crear presets!!)
    def getConfig(self):
        return (self.ampS.get(),self.ratioS.get(),self.betaS.get(),
                self.attackS.get(), self.decayS.get(), self.sustainS.get(),
                self.releaseS.get())

    # activación de nota
    def noteOn(self,midiNote):
        # si está el dict de canales apagamos nota actual con envolvente de fadeout
        # y guardamos en tails. El next devolverá este tail y luego comenzará la nota
        if midiNote in self.channels:                   
            lastAmp = self.channels[midiNote].adsr.last # ultimo valor de la envolvente: inicio del fadeOut
            env = Env([(0,lastAmp),(CHUNK,0)]).next()   # envolvente             
            signal = self.channels[midiNote].next()     # señal          
            self.tails[midiNote] = env*signal           # diccionario de tails (notas apagadas) 

        # generamos un nuevo synth en un canal indexado con notaMidi
        # con los parámetros actuales del synth
        freq= freqsMidi[midiNote]
        self.channels[midiNote]= SynthFM(
                fc=freq,
                amp=self.ampS.get(), ratio=self.ratioS.get(), beta=self.betaS.get(),
                attack = self.attackS.get(), decay= self.decayS.get(),
                sustain=self.sustainS.get(), release=self.releaseS.get())

    # apagar nota -> propagamos noteOff al synth, que se encargará de hacer el release
    def noteOff(self,midiNote):
        if midiNote in self.channels: # está el dict, release
            self.channels[midiNote].noteOff()


    # lectura de teclas de teclado como eventos tkinter
    def down(self,event):
        c = event.keysym

        # tecla "panic" -> apagamos todos los sintes de golpe!
        if c=='0': 
            self.stop()            
        elif c in teclas:
            midiNote = 48+teclas.index(c) # buscamos indice y trasnportamos a C3 (48 en midi)        
            print(f'noteOn {midiNote}')
            self.noteOn(midiNote)         # arrancamos noteOn con el instrumento 
            

    def up(self,event):
        c = event.keysym
        if c in teclas:
            midiNote = 48+teclas.index(c) # buscamos indice y hacemos el noteOff
            print(f'noteOff {midiNote}')
            self.noteOff(midiNote)

    # siguiente chunck del generador: sumamos señal de canales y hacemos limpia de silenciados
    def next(self):
        out = np.zeros(CHUNK)          
        for c in list(self.channels):            # convertimos las keys a lista para mantener la lista de claves original
            if self.channels[c].state == 'off':  # si no, modificamos diccionario en el bucle de recorrido de claves -> error 
                del self.channels[c]
            else: # si la nota está el diccionario de tails devolvemos el fadeout generado en noteOn y elminamos tail
                if c in self.tails:                  
                    out += self.tails[c]
                    del self.tails[c]
                else:
                    out += self.channels[c].next()
        return out        

    # boton del pánico para apagar todas las notas de golpe
    def stop(self):
        self.channels = dict() # delegamos en el garbage collector
        # for c in list(self.channels): del self.channels[c]



In [ ]:
# prueba de instrumento

import sys
sys.path.insert(0, "./files")        


from instrument import *
import sounddevice as sd
import matplotlib.pyplot as plt
from tkinter import *


tk = Tk()
ins = Instrument(tk)
ins.amp = 1
ins.sustain=0.8

ins.noteOn(60)
signal = np.zeros(0)
for i in range(25):
    signal = np.concatenate((signal,ins.next()))

N = 1024
s = signal[-N:]

ins.noteOn(70)
signal2 = np.zeros(0)
for i in range(25):
    signal2 = np.concatenate((signal2,ins.next()))

#s = np.concatenate((s,signal2[:N]))

tot = np.concatenate((signal,signal2))
sd.play(tot)

plt.plot(s)    
tk.mainloop()

# ignorar error!!


# Enrutado de la señal del intrumento para controlarlo en tiempo real

- Enrutamos la señal del instrumento a la entrada del stream de *sounddevice*

- Generalizamos el *callback* de sounddevice: el input ahora va a admitir una lista de señales y hará la *mezcla* (suma) de todas ellas

    - Visto de otro modo: todas las señales que queramos que suenen se incluyen en la lista **inputs**

- En el callback se pide un **next()** de todas las entradas de **input**, se suman y se mandan al output


### Un último detalle para capturar la liberación de la tecla

- El sistema operativo por defecto tiene activado el *modo repetición* de tecla: cuando se mantiene una tecla pulsada más de *x* milisegundos se autopulsa automáticamente.

- Por facilidad de uso desactivamos esta opción desde Python (llamando al SO):

    ```os.system('xset r off')```

- Al terminar reactivamos la opción:

    ```os.system('xset r on')```

In [ ]:
import sys
sys.path.insert(0, "./files")        


from tkinter import *
import os
from instrument import *
import sounddevice as sd


def test():
    def callback(outdata, frames, time, status):    
        if status: print(status)    
        s = np.sum([i.next() for i in inputs],axis=0)
        s = np.float32(s)
        outdata[:] = s.reshape(-1, 1)

    tk = Tk()
    ins = Instrument(tk)
    inputs = [ins]    

    # desactivar repeticion de teclas
    os.system('xset r off')

    stream = sd.OutputStream(samplerate=SRATE, channels=1, blocksize=CHUNK, callback=callback)    
    stream.start()
    tk.mainloop()

    # reactivar repeticion de teclas   
    os.system('xset r on')
    stream.close()

test()    


## Hebras de ejecución

En el ejemplo anterior tenemos 2 hebras que controlan la ejecución:

- *stream* está en modo *callback* escuchando/esperando señal en una hebra
    - Escucha todas las señales/generadores que hay en la lista *inputs*
    - Hace la mezcla de esas señales y las reproduce    
    - Todo esto lo hace por CHUNKs

- *tk.mainloop()* toma el control de la ejecución de la aplicación -> el queda en la ventana de ejecución 
    - Escucha eventos de los widgets (botones, sliders, etc)

*TkInter* tiene el control de ejecución... ya no implementamos bucles (será importante en breve)
  